![Banner](../Image/01_DeepCuaslaML.png)

# 1.5 CEVAE: Generative Model with Latent Confounders

> **Note:** CEVAE requires **PyTorch**. The `CEVAE` estimator in `pydeepcausalml.effect` learns latent confounders from proxy covariates via a VAE.

Generative models for causal inference have become a powerful framework for estimating treatment effects in complex, high-dimensional settings. Among them, the **Counterfactual Variational Autoencoder (CEVAE)** stands out for its ability to handle *latent confounding* - a pervasive challenge in observational data where the true confounders are not directly measured but are only accessible through noisy proxy variables. By embedding causal assumptions directly into a deep generative model, CEVAE learns a structured latent representation that supports principled counterfactual reasoning and accurate individual treatment effect estimation.


## Overview

CEVAE is a deep generative model for causal inference introduced by Louizos et al. (2017). Its central insight is that in many real-world settings the observed covariates **X** are not the true confounders - they are noisy proxies of some underlying latent variable **Z** that simultaneously influences both treatment assignment and outcomes. Standard causal methods that condition on **X** directly therefore fail to fully eliminate confounding. CEVAE addresses this by learning **Z** from data using a Variational Autoencoder (VAE) framework and then performing causal inference in that latent space.

### The Causal Problem CEVAE Solves

Methods such as `TARNet`, `CFRNet`, `DragonNet`, and `GANITE` all rely on the standard *ignorability* assumption - that all confounders are observed in **X**. In practice this is rarely satisfied. A patient's true health status **Z** is never fully measured; the clinician sees only noisy proxies such as blood pressure readings, laboratory values, and clinical notes. Conditioning on these proxies as if they were **Z** leaves residual confounding that biases causal estimates.

CEVAE (Louizos et al., NeurIPS 2017) takes a different approach: it uses a VAE to *infer* the latent confounder **Z** from the proxy measurements, and then trains a causal model on top of this inferred representation. Treatment and outcome information are fed into the encoder during training, giving the model every available signal to disentangle the underlying confounder from measurement noise.

In the standard potential outcomes setup, we observe:

-   **X** - proxy covariates (e.g., test scores, survey responses, clinical measurements)
-   **T** - binary treatment indicator
-   **Y** - observed outcome

The key assumption is that there exists a latent **Z** such that:

$$T \perp Y(t) \mid Z \quad \text{(ignorability holds in the latent space)}$$

but *not* necessarily $T \perp Y(t) \mid X$. Because **X** is an imperfect, noisy measurement of the true confounder **Z**, conditioning only on **X** leaves residual bias. CEVAE recovers **Z** and conditions on it instead.

### The Generative Model

CEVAE specifies a joint generative process over all observed variables, driven by the latent confounder **Z**:

$$p(Z) = \mathcal{N}(0, I)$$

$$p(X \mid Z) = \prod_j p(x_j \mid Z) \quad \text{(factored, mixed likelihood)}$$

$$p(T \mid Z) = \text{Bernoulli}\!\left(\sigma(f_T(Z))\right)$$

$$p(Y \mid T, Z) = p(Y \mid T=0, Z)\cdot(1-T) + p(Y \mid T=1, Z)\cdot T$$

where $f_T$ and the outcome distributions are parameterized by neural networks. The outcome likelihood is Gaussian for continuous outcomes and Bernoulli for binary outcomes. Critically, **T and Y are conditionally independent given Z**, which enforces the ignorability assumption in the latent space.

### The Inference Model (Encoder)

Because **Z** is unobserved, CEVAE must infer it from data. It defines a variational posterior

$$q(Z \mid X, T, Y) \approx p(Z \mid X, T, Y)$$

implemented as a neural network that takes all observed variables as input and outputs the parameters $(\mu_Z, \sigma_Z^2)$ of a Gaussian approximate posterior. Feeding treatment and outcome into the encoder during training allows the model to use all available signal about the latent confounder. The reparameterization trick makes end-to-end backpropagation through stochastic **Z** samples possible.

### Training via the ELBO

CEVAE is trained by maximizing the Evidence Lower Bound (ELBO), which balances reconstruction fidelity against regularization of the latent space:

$$\mathcal{L} = \mathbb{E}_{q(Z|X,T,Y)}\!\left[\log p(X|Z) + \log p(T|Z) + \log p(Y|T,Z)\right] - D_{\mathrm{KL}}\!\left(q(Z|X,T,Y) \,\|\, p(Z)\right)$$

The **reconstruction terms** ensure the generative model explains the observed data well. The **KL divergence** regularizes **Z** toward the unit Gaussian prior, preventing overfitting and encouraging a smooth, well-structured latent space. Optimizing this objective jointly trains the encoder and all decoder heads.

### Counterfactual Prediction

Once trained, CEVAE estimates individual treatment effects (ITE) through a three-step procedure:

1.  **Encode**: given $(X_i, T_i, Y_i)$, sample $Z_i \sim q(Z \mid X_i, T_i, Y_i)$
2.  **Predict both potential outcomes**: $\hat{Y}_i(0) = \mathbb{E}[Y \mid T{=}0, Z_i]$ and $\hat{Y}_i(1) = \mathbb{E}[Y \mid T{=}1, Z_i]$
3.  **Estimate ITE**: $\hat{\tau}_i = \hat{Y}_i(1) - \hat{Y}_i(0)$

Both potential outcomes are predicted from the same inferred **Z**, which is what gives CEVAE its *counterfactual coherence* — the factual and counterfactual are grounded in the same underlying confounder estimate.


### Key Architectural Components

| Component | Role |
|------------------------------------|------------------------------------|
| **Encoder** $q(Z \mid X, T, Y)$ | Infers the latent confounder from all observed variables |
| **Decoder** $p(X \mid Z)$ | Reconstructs proxy covariates; anchors the latent space to observables |
| **Treatment head** $p(T \mid Z)$ | Models the propensity score in latent space |
| **Outcome heads** $p(Y \mid T{=}0, Z),\ p(Y \mid T{=}1, Z)$ | Separate networks for each potential outcome |
| **Prior** $p(Z)$ | Standard Gaussian regularizer |

### CEVAE Model Architecture

The figure below illustrates the full data flow through CEVAE. The left side shows the *inference path* (encoder), and the right side shows the *generative path* (decoder and prediction heads).


![](../Image/CEVAE_arcitecture.png)

### How CEVAE Relates to Other Methods

| Method | Handles latent confounders | Architecture |
|------------------------|------------------------|------------------------|
| TARNet / CFRNet | No — assumes **X** contains all confounders | Shared encoder + two outcome heads |
| DragonNet | No — same assumption; adds a propensity head | Shared encoder + propensity + outcome heads |
| **CEVAE** | **Yes** — explicitly models **Z** as latent | Full VAE generative model |

CEVAE is strictly more general in its confounding assumptions. The trade-off is that it is harder to train, more sensitive to generative model misspecification, and computationally heavier than discriminative approaches such as TARNet.



### Assumptions and Limitations

**Assumptions:**

-   Ignorability holds in the latent space **Z**, not necessarily in **X**
-   The generative model structure (the way **Z** produces **X**, **T**, **Y**) is correctly specified
-   The variational posterior approximation is sufficiently flexible

**Practical limitations:**

-   Sensitive to the choice of decoder architecture for **X** — misspecifying $p(X \mid Z)$ can corrupt the inferred **Z**
-   Identifiability of **Z** is not guaranteed in general; the structured causal graph helps anchor the latent space, but cannot guarantee it
-   Harder to train stably than TARNet or CFRNet, especially on small datasets
-   Counterfactual quality depends on how well the encoder separates confounding signal from measurement noise in **X**



### Extensions

Subsequent work has refined and extended the CEVAE framework:

-   **DCEVAE** disentangles the latent space into components that are *independent* of treatment versus *correlated* with it, improving counterfactual generation quality. See [ojs.aaai.org](https://ojs.aaai.org/index.php/AAAI/article/view/16990) for details.
-   **VCI (Variational Causal Inference)** argues that the vanilla CEVAE formulation is not fully aligned with Pearl's counterfactual semantics, and proposes a stricter framework with disentangled exogenous noise abduction — particularly important for high-dimensional outcomes such as gene expression profiles. See [arxiv.org](https://arxiv.org/html/2410.12730v2) for the full treatment.

The primary limitation shared by all these approaches is that **X** must be sufficiently informative about **Z** to allow reliable recovery. When proxy measurements are very weak, the confounder posterior will be poorly identified and causal estimates will degrade.

## Implementation in Python

We use `pydeepcausalml.effect.CEVAE` on the IHDP semi-synthetic dataset and a synthetic hidden-confounder benchmark, comparing against meta-learners.

### Check and Install Required Python Packages

The following packages are used in this notebook. Install any missing packages with the cell below:

`numpy`, `pandas`, `scipy`, `torch`, `scikit-learn`, `matplotlib`, `seaborn`, `causaldata`, `pydeepcausalml`

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

PACKAGES = [
    "numpy", "pandas", "scipy", "torch", "scikit-learn",
    "matplotlib", "seaborn", "causaldata",
]

for pkg in PACKAGES:
    mod = "sklearn" if pkg == "scikit-learn" else pkg
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import pydeepcausalml  # noqa: F401
except ImportError:
    repo_root = Path("..").resolve()
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root)])

print("Packages ready.")

### Verify imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pydeepcausalml import set_seed

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, LogisticRegression, LinearRegression
from sklearn.model_selection import KFold


def load_nsw_mixtape() -> pd.DataFrame:
    """Load NSW (Lalonde) mixtape data (same variables as causaldata::nsw_mixtape)."""
    try:
        import causaldata.nsw_mixtape.data as nsw_data
        dta = Path(nsw_data.__file__).parent / "nsw_mixtape.dta"
        return pd.read_stata(dta)
    except Exception:
        pass
    url = (
        "https://raw.githubusercontent.com/NickCH-K/causaldata/master/"
        "data-raw/nsw_mixtape.dta"
    )
    return pd.read_stata(url)


def propensity_elastic_net(X: np.ndarray, treatment: np.ndarray) -> np.ndarray:
    model = LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=2000
    )
    model.fit(X, treatment)
    return model.predict_proba(X)[:, 1]


class SLearner:
    def __init__(self, learner: str = "ranger"):
        self.learner = learner

    def fit(self, X, t, y):
        Xt = np.column_stack([X, t])
        if self.learner == "lm":
            self.model_ = LinearRegression().fit(Xt, y)
        else:
            self.model_ = RandomForestRegressor(
                n_estimators=100, random_state=42, n_jobs=-1
            ).fit(Xt, y)
        return self

    def predict(self, X):
        y0 = self.model_.predict(np.column_stack([X, np.zeros(len(X))]))
        y1 = self.model_.predict(np.column_stack([X, np.ones(len(X))]))
        return y1 - y0


class TLearner:
    def __init__(self, learner: str = "ranger"):
        self.learner = learner

    def fit(self, X, t, y):
        t = t.astype(int)
        if self.learner == "lm":
            self.m0_ = LinearRegression().fit(X[t == 0], y[t == 0])
            self.m1_ = LinearRegression().fit(X[t == 1], y[t == 1])
        else:
            self.m0_ = RandomForestRegressor(
                n_estimators=100, random_state=42, n_jobs=-1
            ).fit(X[t == 0], y[t == 0])
            self.m1_ = RandomForestRegressor(
                n_estimators=100, random_state=42, n_jobs=-1
            ).fit(X[t == 1], y[t == 1])
        return self

    def predict(self, X):
        return self.m1_.predict(X) - self.m0_.predict(X)


class XLearner:
    def __init__(self, learner: str = "ranger"):
        self.learner = learner

    def _reg(self):
        if self.learner == "lm":
            return LinearRegression()
        return RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

    def fit(self, X, t, y, p=None):
        t = t.astype(int)
        self.propensity_model_ = LogisticRegression(
            penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=2000
        ).fit(X, t)
        m0, m1 = self._reg(), self._reg()
        m0.fit(X[t == 0], y[t == 0])
        m1.fit(X[t == 1], y[t == 1])
        d1 = y[t == 1] - m0.predict(X[t == 1])
        d0 = m1.predict(X[t == 0]) - y[t == 0]
        tau1 = self._reg().fit(X[t == 1], d1)
        tau0 = self._reg().fit(X[t == 0], d0)
        self.tau0_, self.tau1_ = tau0, tau1
        return self

    def predict(self, X):
        p = self.propensity_model_.predict_proba(X)[:, 1]
        return p * self.tau0_.predict(X) + (1 - p) * self.tau1_.predict(X)


class RLearner:
    def __init__(self, learner: str = "ranger", n_fold: int = 3):
        self.learner = learner
        self.n_fold = n_fold

    def _reg(self):
        if self.learner == "lm":
            return LinearRegression()
        return RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

    def fit(self, X, t, y, p=None):
        t = t.astype(int).astype(float)
        p = propensity_elastic_net(X, t) if p is None else p
        e = t - p
        m = self._reg()
        kf = KFold(n_splits=self.n_fold, shuffle=True, random_state=42)
        y_hat = np.zeros_like(y, dtype=float)
        for tr, va in kf.split(X):
            m.fit(X[tr], y[tr])
            y_hat[va] = m.predict(X[va])
        resid = y - y_hat
        tau = self._reg()
        tau.fit(X, resid / (e + 1e-6), sample_weight=e**2)
        self.tau_ = tau
        return self

    def predict(self, X):
        return self.tau_.predict(X)


def get_cumgain(df_preds, outcome_col="y", treatment_col="w", treatment_effect_col="tau", normalize=True):
    """Cumulative gain curve (CausalML-style AUUC helper)."""
    y = df_preds[outcome_col].to_numpy()
    w = df_preds[treatment_col].to_numpy()
    tau = df_preds[treatment_effect_col].to_numpy()
    model_cols = [c for c in df_preds.columns if c not in {outcome_col, treatment_col, treatment_effect_col}]
    n = len(df_preds)
    out = {}
    for col in model_cols:
        scores = df_preds[col].to_numpy()
        order = np.argsort(-scores)
        cum = []
        treated_so_far = control_so_far = 0.0
        for i, idx in enumerate(order, start=1):
            if w[idx] == 1:
                treated_so_far += y[idx]
            else:
                control_so_far += y[idx]
            lift = treated_so_far / max((w[order[:i]] == 1).sum(), 1) - control_so_far / max(
                (w[order[:i]] == 0).sum(), 1
            )
            cum.append(lift)
        curve = np.array(cum)
        if normalize and curve[-1] != 0:
            curve = curve / np.abs(curve[-1])
        out[col] = curve
    out[treatment_effect_col] = out.get(treatment_effect_col, out[model_cols[0]])
    return pd.DataFrame(out)


def plot_gain(df_preds, outcome_col="y", treatment_col="w", treatment_effect_col="tau", model_cols=None, main="", normalize=True):
    import matplotlib.pyplot as plt

    gain = get_cumgain(df_preds, outcome_col, treatment_col, treatment_effect_col, normalize)
    model_cols = model_cols or [c for c in gain.columns]
    plt.figure(figsize=(8, 5))
    for col in model_cols:
        if col in gain.columns:
            plt.plot(gain[col].values, label=col)
    plt.xlabel("Population fraction targeted")
    plt.ylabel("Cumulative gain" + (" (normalized)" if normalize else ""))
    plt.title(main)
    plt.legend()
    plt.tight_layout()
    plt.show()


def load_ihdp(replications: int = 2, random_state: int = 1):
    base_url = "https://raw.githubusercontent.com/uber/causalml/master/docs/examples/data"
    cols = ["treatment", "y_factual", "y_cfactual", "mu0", "mu1"] + [f"X{i}" for i in range(25)]
    parts = []
    for i in range(1, 10):
        url = f"{base_url}/ihdp_npci_{i}.csv"
        tmp = pd.read_csv(url, header=None)
        tmp.columns = cols[: tmp.shape[1]]
        parts.append(tmp)
    df = pd.concat(parts, ignore_index=True)
    if replications > 1:
        df = pd.concat([df] * replications, ignore_index=True)
    binfeats = [f"X{i}" for i in range(6, 25)]
    contfeats = [f"X{i}" for i in range(6)]
    xcols = binfeats + contfeats
    X = df[xcols].to_numpy(dtype=float)
    treatment = df["treatment"].to_numpy(dtype=int)
    y = df["y_factual"].to_numpy(dtype=float)
    tau = np.where(
        treatment == 1,
        df["y_factual"] - df["y_cfactual"],
        df["y_cfactual"] - df["y_factual"],
    )
    n = len(X)
    rng = np.random.default_rng(random_state)
    val_idx = rng.choice(n, size=int(0.2 * n), replace=False)
    train_idx = np.setdiff1d(np.arange(n), val_idx)
    return df, X, treatment, y, tau, train_idx, val_idx


def simulate_hidden_confounder(n=2000, p=5, sigma=1.0, adj=0, random_state=42):
    """Synthetic DGP with latent confounder (CausalML-style benchmark)."""
    rng = np.random.default_rng(random_state)
    z = rng.normal(size=n)
    X = np.column_stack([z + rng.normal(scale=sigma, size=n) for _ in range(p)])
    e = 1 / (1 + np.exp(-(adj + 0.5 * z)))
    w = rng.binomial(1, e)
    b = 2 * z + rng.normal(scale=sigma, size=n)
    tau = 0.5 + 0.3 * z
    y = b + w * tau + rng.normal(scale=sigma, size=n)
    return {"X": X, "y": y, "w": w, "tau": tau, "b": b, "e": e}


def result_table(df_preds, tau_true):
    model_cols = [c for c in ["S", "T", "X", "R", "GANITE", "CEVAE"] if c in df_preds.columns]
    methods = model_cols + ["actual"]
    ate = [df_preds[m].mean() for m in model_cols] + [float(tau_true.mean())]
    mae = [float(np.mean(np.abs(df_preds[m] - tau_true))) for m in model_cols] + [np.nan]
    gain = get_cumgain(df_preds)
    auuc = [float(gain[m].mean()) if m in gain.columns else np.nan for m in model_cols]
    auuc.append(float(gain["tau"].mean()) if "tau" in gain.columns else np.nan)
    return pd.DataFrame({"Method": methods, "ATE": ate, "MAE": mae, "AUUC": auuc})


def synthetic_summary(preds, tau_ref, actual_ate):
    rows = []
    for name, p in preds.items():
        rows.append({
            "Method": name,
            "ATE": p.mean(),
            "MSE": np.mean((p - tau_ref) ** 2),
            "AbsPctErrorATE": abs(p.mean() / actual_ate - 1) if actual_ate != 0 else np.nan,
        })
    rows.append({"Method": "Actuals", "ATE": actual_ate, "MSE": np.nan, "AbsPctErrorATE": np.nan})
    return pd.DataFrame(rows)

import os
from pydeepcausalml.effect import CEVAE

RUN_FAST = os.getenv("PYDEEPCAUSALML_FAST_RENDER", "true").lower() == "true"
set_seed(42)

## Fitting CEVAE on the IHDP Dataset (semi-synthetic)

In [ ]:
df, X, treatment, y, tau, train_idx, val_idx = load_ihdp(
    replications=2 if RUN_FAST else 10, random_state=1
)
X_train, X_val = X[train_idx], X[val_idx]
treatment_train, treatment_val = treatment[train_idx], treatment[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
tau_train, tau_val = tau[train_idx], tau[val_idx]
print("Train size:", len(train_idx), "| Val size:", len(val_idx))

### Fitting the CEVAE Model

In [ ]:
n_samp_pred = 1 if RUN_FAST else 100
cv = CEVAE(
    latent_dim=10 if RUN_FAST else 20,
    hidden_dim=64 if RUN_FAST else 200,
    num_layers=2 if RUN_FAST else 3,
    num_samples=20 if RUN_FAST else 1000,
    epochs=10 if RUN_FAST else 50,
    batch_size=256 if RUN_FAST else 100,
    random_state=42,
    verbose=not RUN_FAST,
    log_every=5,
).fit(X_train, treatment_train, y_train)

ite_train_cevae = cv.predict_cate(X_train, num_samples=n_samp_pred)
ite_val_cevae = cv.predict_cate(X_val, num_samples=n_samp_pred)
print("CEVAE — ATE (train):", round(ite_train_cevae.mean(), 4),
      "| ATE (val):", round(ite_val_cevae.mean(), 4))

### Fitting Meta-Learners

In [ ]:
p_train = propensity_elastic_net(X_train, treatment_train)
n_fold_meta = 3 if RUN_FAST else 5
sl = SLearner("ranger").fit(X_train, treatment_train, y_train)
tl = TLearner("ranger").fit(X_train, treatment_train, y_train)
xl = XLearner("ranger").fit(X_train, treatment_train, y_train, p=p_train)
rl = RLearner("ranger", n_fold=n_fold_meta).fit(X_train, treatment_train, y_train, p=p_train)

### Results: Training Set

In [ ]:
df_preds_train = pd.DataFrame({
    "S": sl.predict(X_train), "T": tl.predict(X_train),
    "X": xl.predict(X_train), "R": rl.predict(X_train),
    "CEVAE": ite_train_cevae, "tau": tau_train,
    "w": treatment_train, "y": y_train,
})
display(result_table(df_preds_train, tau_train).round(4))
plot_gain(df_preds_train, model_cols=["S", "T", "X", "R", "CEVAE", "tau"],
          main="Cumulative gain — training set")

### Results: Validation Set

In [ ]:
df_preds_val = pd.DataFrame({
    "S": sl.predict(X_val), "T": tl.predict(X_val),
    "X": xl.predict(X_val), "R": rl.predict(X_val),
    "CEVAE": ite_val_cevae, "tau": tau_val,
    "w": treatment_val, "y": y_val,
})
display(result_table(df_preds_val, tau_val).round(4))
plot_gain(df_preds_val, model_cols=["S", "T", "X", "R", "CEVAE", "tau"],
          main="Cumulative gain — validation set")

## Fitting CEVAE on a Synthetic Dataset with Hidden Confounding

In [ ]:
n_syn = 2000 if RUN_FAST else 10000
d_syn = simulate_hidden_confounder(n=n_syn, p=5, sigma=1.0, adj=0)
X_syn, y_syn, w_syn, tau_syn = d_syn["X"], d_syn["y"], d_syn["w"], d_syn["tau"]
rng = np.random.default_rng(123)
val_idx = rng.choice(len(X_syn), size=int(0.2 * len(X_syn)), replace=False)
train_idx = np.setdiff1d(np.arange(len(X_syn)), val_idx)
X_syn_tr, X_syn_val = X_syn[train_idx], X_syn[val_idx]
y_syn_tr, y_syn_val = y_syn[train_idx], y_syn[val_idx]
w_syn_tr, w_syn_val = w_syn[train_idx], w_syn[val_idx]
tau_syn_tr, tau_syn_val = tau_syn[train_idx], tau_syn[val_idx]
p_syn_tr = propensity_elastic_net(X_syn_tr, w_syn_tr)

In [ ]:
n_samp_syn = 1 if RUN_FAST else 100
cv_syn = CEVAE(
    latent_dim=10 if RUN_FAST else 20,
    hidden_dim=64 if RUN_FAST else 200,
    num_layers=2 if RUN_FAST else 3,
    epochs=8 if RUN_FAST else 50,
    batch_size=256 if RUN_FAST else 100,
    random_state=42,
).fit(X_syn_tr, w_syn_tr, y_syn_tr)

preds_syn_train = {"CEVAE": cv_syn.predict_cate(X_syn_tr, num_samples=n_samp_syn)}
preds_syn_valid = {"CEVAE": cv_syn.predict_cate(X_syn_val, num_samples=n_samp_syn)}
for learner_name in ["lm", "ranger"]:
    sl_s = SLearner(learner_name).fit(X_syn_tr, w_syn_tr, y_syn_tr)
    tl_s = TLearner(learner_name).fit(X_syn_tr, w_syn_tr, y_syn_tr)
    xl_s = XLearner(learner_name).fit(X_syn_tr, w_syn_tr, y_syn_tr, p=p_syn_tr)
    rl_s = RLearner(learner_name, n_fold=n_fold_meta).fit(X_syn_tr, w_syn_tr, y_syn_tr, p=p_syn_tr)
    for meta, est in [("S", sl_s), ("T", tl_s), ("X", xl_s), ("R", rl_s)]:
        label = f"{meta} ({learner_name})"
        preds_syn_train[label] = est.predict(X_syn_tr)
        preds_syn_valid[label] = est.predict(X_syn_val)

print("Synthetic — training summary:")
display(synthetic_summary(preds_syn_train, tau_syn_tr, tau_syn_tr.mean()).round(4))
print("Synthetic — validation summary:")
display(synthetic_summary(preds_syn_valid, tau_syn_val, tau_syn_val.mean()).round(4))

## Summary and Conclusions

CEVAE embeds causal assumptions in a generative VAE to recover latent confounders from proxy covariates.


## References

- [CEVAE vs. Meta-Learners Benchmark (CausalML)](https://causalml.readthedocs.io/en/latest/examples/cevae_example.html)
- Louizos et al. (2017). [Causal Effect Inference with Deep Latent Variable Models](http://papers.nips.cc/paper/7223-causal-effect-inference-with-deep-latent-variable-models.pdf)
- [PyDeepCausalML](https://github.com/zia207/PyDeepCausalML)